In [1]:
def dy_dx(x):
  return 2*x

In [2]:
dy_dx(3)

6

In [72]:
# Import the PyTorch library
import torch

# Create a tensor 'x' with the value 3.0 and enable gradient tracking
x = torch.tensor(3.0, requires_grad=True)

# Define a function 'y' as the square of x
y = x**2

# Display the value of x (optional, typically used in notebooks)
print(x)

# Display the value of y, which is x squared (optional)
print(y)

# Perform backpropagation to compute the gradient of y with respect to x
print(y.backward())

# Print the gradient of y with respect to x (i.e., dy/dx = 2*x)
print(x.grad)


tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
None
tensor(6.)


In [73]:
# Import the math module for standard Python math functions
import math

# Define a Python function to compute the derivative dz/dx manually:
# z = sin(x^2), so dz/dx = 2x * cos(x^2)
def dz_dx(x):
    return 2 * x * math.cos(x**2)

# Call the manual derivative function with x = 4
dz_dx(4)

# Create a PyTorch tensor 'x' with value 4.0 and enable gradient tracking
x = torch.tensor(4.0, requires_grad=True)

# Compute y = x^2
y = x ** 2

# Compute z = sin(y) = sin(x^2)
z = torch.sin(y)

# Display intermediate values (mostly useful in notebooks)
print(x)
print(y)
print(z)

# Perform backpropagation to compute dz/dx using autograd
print(z.backward())

# Access the gradient of z with respect to x (i.e., dz/dx)
print(x.grad)

# Accessing y.grad will return None because y is not a leaf tensor
# (PyTorch does not retain gradients for intermediate tensors by default)
print(y.grad)


tensor(4., requires_grad=True)
tensor(16., grad_fn=<PowBackward0>)
tensor(-0.2879, grad_fn=<SinBackward0>)
None
tensor(-7.6613)
None


<ipython-input-73-24d08828ab5b>:34: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(y.grad)


> 🔍 Note: If you want to get the gradient of `y`, you'd need to call .`retain_grad()` on it before `backward()` like this:

In [74]:
import torch

# Input feature value (scalar)
x = torch.tensor(6.7)  # Example input

# True label (binary classification: 0 or 1)
y = torch.tensor(0.0)  # Ground truth label

# Model parameters (scalar values for simplicity)
w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

# Custom function to compute Binary Cross-Entropy Loss for a single prediction
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # Small constant to avoid log(0) which would cause NaN
    # Clamp prediction to stay within (epsilon, 1 - epsilon) range
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    # Compute BCE loss using the formula:
    # -[y*log(p) + (1-y)*log(1-p)]
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

# Forward pass: compute linear combination (z = w*x + b)
z = w * x + b  # Linear transformation

# Apply sigmoid activation to get the predicted probability (between 0 and 1)
y_pred = torch.sigmoid(z)

# Compute binary cross-entropy loss between prediction and true label
loss = binary_cross_entropy_loss(y_pred, y)

# Output the computed loss value
print(f"Computed Loss: {loss}")


Computed Loss: 6.701176166534424


In [75]:
# Derivatives for backpropagation using the chain rule

# 1. dL/d(y_pred): Derivative of Binary Cross-Entropy loss w.r.t. prediction
# This comes from the derivative of the BCE loss:
# dL/d(y_pred) = (y_pred - y) / (y_pred * (1 - y_pred))
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz: Derivative of the sigmoid function w.r.t. its input z
# sigmoid'(z) = y_pred * (1 - y_pred)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: Partial derivatives of z w.r.t. parameters
dz_dw = x  # Since z = w * x + b => dz/dw = x
dz_db = 1  # Since dz/db = 1 (bias has a direct linear contribution)

# Combine using chain rule:
# dL/dw = dL/d(y_pred) * d(y_pred)/dz * dz/dw
dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw

# dL/db = dL/d(y_pred) * d(y_pred)/dz * dz/db
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

# Print the manually computed gradients
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")


Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


> ✅ Summary: This is a manual backpropagation through a single-layer logistic regression model. It shows how the loss gradient propagates backward from prediction to each parameter (`w`, `b`) using the chain rule.

In [77]:
import torch

# Input feature and true label (both scalar)
x = torch.tensor(6.7)
y = torch.tensor(0.0)

# Initialize model parameters with gradient tracking enabled
w = torch.tensor(1.0, requires_grad=True)  # Weight
b = torch.tensor(0.0, requires_grad=True)  # Bias

# Print input and initial parameter values
print(f"x: {x}")
print(f"y (true label): {y}")
print(f"Initial weight w: {w}")
print(f"Initial bias b: {b}")

# Forward pass: compute linear combination
z = w * x + b  # z = w*x + b
print(f"z (linear output): {z}")

# Apply sigmoid activation to get predicted probability
y_pred = torch.sigmoid(z)
print(f"y_pred (after sigmoid): {y_pred}")

# Binary Cross-Entropy Loss for a single prediction
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # Small value to prevent log(0)
    # Clamp predictions to avoid numerical instability
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    # Compute the BCE loss
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

# Compute the loss between prediction and true label
loss = binary_cross_entropy_loss(y_pred, y)
print(f"Binary Cross-Entropy Loss: {loss}")

# Backward pass: compute gradients of loss w.r.t. model parameters
loss.backward()

# Print the gradients computed by autograd
print(f"Gradient of loss w.r.t weight (w.grad): {w.grad}")  # ∂loss/∂w
print(f"Gradient of loss w.r.t bias (b.grad): {b.grad}")    # ∂loss/∂b


x: 6.699999809265137
y (true label): 0.0
Initial weight w: 1.0
Initial bias b: 0.0
z (linear output): 6.699999809265137
y_pred (after sigmoid): 0.998770534992218
Binary Cross-Entropy Loss: 6.701176166534424
Gradient of loss w.r.t weight (w.grad): 6.6917619705200195
Gradient of loss w.r.t bias (b.grad): 0.9987704753875732


✅ Explanation:
PyTorch automatically applies the chain rule to compute gradients of the loss with respect to `w` and `b`. These match the manually derived gradients from your earlier code.

In [78]:
import torch

# --- PART 1: Compute gradient for a vector ---
# Create a tensor with gradient tracking enabled
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"x: {x}")

# Compute mean of squared elements: y = mean(x^2)
y = (x**2).mean()
print(f"y = mean(x^2): {y}")

# Perform backpropagation to compute dy/dx
y.backward()

# Print gradients: dy/dx = 2x / len(x)
print(f"Gradient of y w.r.t x: {x.grad}")

# --- PART 2: Clear and recompute gradient for scalar x ---
# Create a new scalar tensor with requires_grad=True
x = torch.tensor(2.0, requires_grad=True)
print(f"\nNew x: {x}")

# Compute y = x^2
y = x ** 2
print(f"y = x^2: {y}")

# Backward pass to compute dy/dx
y.backward()

# Print gradient: dy/dx = 2x = 4.0
print(f"Gradient of y w.r.t x: {x.grad}")

# Zero the gradient manually (useful when accumulating gradients in loops)
x.grad.zero_()
print(f"Gradient after zeroing: {x.grad}")

# --- PART 3: Disabling gradient tracking ---
# Create another scalar tensor with gradient tracking enabled
x = torch.tensor(2.0, requires_grad=True)
print(f"\nAnother new x: {x}")

# Compute y = x^2 again
y = x ** 2
print(f"y = x^2: {y}")

# Backward pass
y.backward()

# Print gradient (still works because requires_grad=True)
print(f"Gradient of y w.r.t x: {x.grad}")


x: tensor([1., 2., 3.], requires_grad=True)
y = mean(x^2): 4.666666507720947
Gradient of y w.r.t x: tensor([0.6667, 1.3333, 2.0000])

New x: 2.0
y = x^2: 4.0
Gradient of y w.r.t x: 4.0
Gradient after zeroing: 0.0

Another new x: 2.0
y = x^2: 4.0
Gradient of y w.r.t x: 4.0


In [79]:
import torch

# --- Option 1: Disable gradient tracking in-place ---
x = torch.tensor(2.0, requires_grad=True)
print(f"Original x: {x} (requires_grad={x.requires_grad})")

# Disable gradient tracking for x (in-place)
x.requires_grad_(False)
print(f"x after requires_grad_(False): {x} (requires_grad={x.requires_grad})")

# Compute y = x^2 with requires_grad=False (no gradient tracking)
y = x ** 2
print(f"y = x^2 with requires_grad=False: {y}")

# Trying to call backward() on y will error because y does not require grad
try:
    y.backward()
except RuntimeError as e:
    print(f"Error calling backward on y with requires_grad=False: {e}")

# --- Option 2: Using detach() to create a new tensor without grad tracking ---
x = torch.tensor(2.0, requires_grad=True)
print(f"\nNew x: {x} (requires_grad={x.requires_grad})")

# Detach creates a tensor sharing storage but without gradient tracking
z = x.detach()
print(f"Detached tensor z: {z} (requires_grad={z.requires_grad})")

# Compute y = x^2 (tracked)
y = x ** 2
print(f"y = x^2 (tracked): {y}")

# Compute y1 = z^2 (not tracked)
y1 = z ** 2
print(f"y1 = z^2 (not tracked): {y1}")

# Backward on y (works fine)
y.backward()
print(f"x.grad after y.backward(): {x.grad}")

# Backward on y1 will error because y1 does not track gradients
try:
    y1.backward()
except RuntimeError as e:
    print(f"Error calling backward on y1 (detached tensor): {e}")

# --- Option 3: Using torch.no_grad() context manager ---
x = torch.tensor(2.0, requires_grad=True)
print(f"\nNew x for no_grad: {x} (requires_grad={x.requires_grad})")

with torch.no_grad():
    y = x ** 2
    print(f"y = x^2 inside torch.no_grad(): {y} (requires_grad={y.requires_grad})")

# Backward on y outside torch.no_grad() will error because y was computed without tracking
try:
    y.backward()
except RuntimeError as e:
    print(f"Error calling backward on y computed inside torch.no_grad(): {e}")


Original x: 2.0 (requires_grad=True)
x after requires_grad_(False): 2.0 (requires_grad=False)
y = x^2 with requires_grad=False: 4.0
Error calling backward on y with requires_grad=False: element 0 of tensors does not require grad and does not have a grad_fn

New x: 2.0 (requires_grad=True)
Detached tensor z: 2.0 (requires_grad=False)
y = x^2 (tracked): 4.0
y1 = z^2 (not tracked): 4.0
x.grad after y.backward(): 4.0
Error calling backward on y1 (detached tensor): element 0 of tensors does not require grad and does not have a grad_fn

New x for no_grad: 2.0 (requires_grad=True)
y = x^2 inside torch.no_grad(): 4.0 (requires_grad=False)
Error calling backward on y computed inside torch.no_grad(): element 0 of tensors does not require grad and does not have a grad_fn
